### This script:
### 1. Loads the Step 3 feature matrix
### 2. Splits into train/test sets (stratified, 80/20)
### 3. Trains three models: Logistic Regression, Random Forest, XGBoost
### 4. Evaluates each with 5-fold cross-validation on the training set
### 5. Evaluates each on the held-out test set
### 6. Produces a comparison table of all models

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")
 
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline        import Pipeline
from sklearn.metrics         import (roc_auc_score, average_precision_score,
                                     f1_score, classification_report,
                                     ConfusionMatrixDisplay)
import xgboost as xgb
import matplotlib
matplotlib.use("Agg")   # non-interactive backend for saving figures
import matplotlib.pyplot as plt

In [ ]:
INPUT_PATH = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs\step3_full.csv"
OUTPUT_DIR = r"C:\Users\USER\PycharmProjects\Non-med-dementia-risk\outputs"
LABEL_COL  = "dementia_binary"
ID_COL     = "NACCID"
RANDOM_STATE = 42
 
os.makedirs(OUTPUT_DIR, exist_ok=True)
 
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"Loaded: {df.shape}")

In [ ]:
# 1. Prepare feature matrix and target
 

feature_cols = [c for c in df.columns if c not in [ID_COL, LABEL_COL]]
X = df[feature_cols].values
y = df[LABEL_COL].values
 
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class balance: {pd.Series(y).value_counts(normalize=True).round(3).to_dict()}")
 

In [ ]:
# 2. Train / test split
#
# Stratified 80/20 split. Stratification preserves the 54.7/45.3 class balance
# in both train and test sets. Since we already reduced to one row per subject
# in Step 2, there is no subject-level leakage risk with a random row split.
 

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
 
print(f"Train size: {X_train.shape[0]} ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Test size:  {X_test.shape[0]} ({X_test.shape[0]/len(X)*100:.1f}%)")
print(f"Train class balance: {pd.Series(y_train).value_counts(normalize=True).round(3).to_dict()}")
print(f"Test class balance:  {pd.Series(y_test).value_counts(normalize=True).round(3).to_dict()}")

In [1]:
# 3. Define models
#
# Three model families chosen to cover different inductive biases:
#
# 1. Logistic Regression — linear baseline, highly interpretable, requires scaling
# 2. Random Forest — ensemble of decision trees, handles non-linearity and
#    feature interactions, no scaling needed
# 3. XGBoost — gradient boosted trees, typically strongest out-of-the-box
#    performance on tabular data, no scaling needed
#
# All use default hyperparameters here. Step 5 will tune each.

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
 
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("clf",    LogisticRegression(
                       max_iter=1000,
                       random_state=RANDOM_STATE,
                       class_weight="balanced"   # handles slight class imbalance
                   ))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        class_weight="balanced"
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        eval_metric="logloss",
        verbosity=0,
        use_label_encoder=False
    ),
}

In [ ]:
# 4. Cross-validation on training set
#
# 5-fold stratified CV gives a robust estimate of generalisation performance
# before touching the test set. Metrics: ROC-AUC, Average Precision (PR-AUC), F1.
 
CV_METRICS = ["roc_auc", "average_precision", "f1"]
 
cv_results = {}
for name, model in models.items():
    print(f"Running CV: {name} ...")
    scores = cross_validate(
        model, X_train, y_train,
        cv=cv,
        scoring=CV_METRICS,
        n_jobs=-1,
        return_train_score=False
    )
    cv_results[name] = {
        "ROC-AUC  (CV mean)": scores["test_roc_auc"].mean().round(4),
        "ROC-AUC  (CV std)":  scores["test_roc_auc"].std().round(4),
        "PR-AUC   (CV mean)": scores["test_average_precision"].mean().round(4),
        "F1       (CV mean)": scores["test_f1"].mean().round(4),
    }
    print(f"  ROC-AUC: {scores['test_roc_auc'].mean():.4f} ± {scores['test_roc_auc'].std():.4f}")
    print(f"  PR-AUC:  {scores['test_average_precision'].mean():.4f}")
    print(f"  F1:      {scores['test_f1'].mean():.4f}\n")
 

In [ ]:
# 5. Train final models on full training set + evaluate on test set

test_results = {}
trained_models = {}
 
for name, model in models.items():
    print(f"Training on full train set: {name} ...")
    model.fit(X_train, y_train)
    trained_models[name] = model
 
    y_pred      = model.predict(X_test)
    y_prob      = model.predict_proba(X_test)[:, 1]
 
    roc_auc     = roc_auc_score(y_test, y_prob)
    pr_auc      = average_precision_score(y_test, y_prob)
    f1          = f1_score(y_test, y_pred)
 
    test_results[name] = {
        "ROC-AUC (test)": round(roc_auc, 4),
        "PR-AUC  (test)": round(pr_auc, 4),
        "F1      (test)": round(f1, 4),
    }
 
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print(f"  PR-AUC:  {pr_auc:.4f}")
    print(f"  F1:      {f1:.4f}")
    print(f"\nClassification Report ({name}):")
    print(classification_report(y_test, y_pred, target_names=["Not at risk", "At risk"]))
 

In [ ]:
# 6. Model comparison table
 
cv_df   = pd.DataFrame(cv_results).T
test_df = pd.DataFrame(test_results).T
comparison = cv_df.join(test_df)
 
print("\n=== Model Comparison ===")
print(comparison.to_string())
 
comparison.to_csv(os.path.join(OUTPUT_DIR, "step4_model_comparison.csv"))
 

In [ ]:
# 7. Confusion matrices
 
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = model.predict(X_test)
    ConfusionMatrixDisplay.from_predictions(
        y_test, y_pred,
        display_labels=["Not at risk", "At risk"],
        ax=ax, colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "step4_confusion_matrices.png"), dpi=150)
print(f"Saved confusion matrices.")

In [ ]:
# 8. ROC curves — all models overlaid
 
from sklearn.metrics import RocCurveDisplay
 
fig, ax = plt.subplots(figsize=(7, 6))
for name, model in trained_models.items():
    y_prob = model.predict_proba(X_test)[:, 1]
    RocCurveDisplay.from_predictions(y_test, y_prob, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_title("ROC Curves — Model Comparison")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "step4_roc_curves.png"), dpi=150)
print(f"Saved ROC curves.")